# Transformer Encoder–Decoder from Scratch (PyTorch)

A tiny, fully explicit implementation: no `nn.Transformer`. Run the cells **top to bottom**.

**Notation**: `B` batch, `S` source length, `T` target length, `D` d_model, `H` heads, `D_HEAD = D/H`, `vocab_size` vocabulary size.
`V` always means the Value tensor.

Pipeline: tokens → IDs → embeddings → +positions → Encoder → Decoder (masked self-attn, cross-attn, FFN) → Linear → logits → softmax → token.

**Install (terminal, once):** `python -m pip install torch`

## 1. Setup and configuration

In [1]:
import math
import torch
import torch.nn as nn

print("torch version:", torch.__version__)
torch.manual_seed(0)              # reproducible random weights

# Toy configuration
B, S, T = 1, 4, 4                 # batch, source length, target length
D, H, D_FF = 8, 2, 16             # d_model, heads, FFN hidden size
D_HEAD = D // H                   # 4: each head works in a D/H-dim subspace
print(f"B={B} S={S} T={T} D={D} H={H} D_HEAD={D_HEAD} D_FF={D_FF}")

torch version: 2.14.1+cu130
B=1 S=4 T=4 D=8 H=2 D_HEAD=4 D_FF=16


## 2. Tiny vocabulary and tokenization
Tokenization = text → tokens. Token ID = integer index in the vocabulary.
Whitespace splitting is a deliberate simplification (real models use subword tokenizers).

In [2]:
vocab = {
    "<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3,
    "i": 4, "love": 5, "cats": 6,
    "j'aime": 7, "les": 8, "chats": 9,
}
id_to_token = {i: t for t, i in vocab.items()}
vocab_size = len(vocab)
PAD_ID, BOS_ID, EOS_ID = vocab["<pad>"], vocab["<bos>"], vocab["<eos>"]

def encode(text, add_bos=False, add_eos=False):
    tokens = text.lower().split()                               # tokenization
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]    # token IDs
    if add_bos: ids = [BOS_ID] + ids
    if add_eos: ids = ids + [EOS_ID]
    return ids

src_ids       = encode("I love cats", add_eos=True)                     # [4, 5, 6, 2]
tgt_full      = encode("j'aime les chats", add_bos=True, add_eos=True)  # [1, 7, 8, 9, 2]
tgt_input_ids = tgt_full[:-1]     # decoder input  (shifted right): [1, 7, 8, 9]
tgt_label_ids = tgt_full[1:]      # labels (what to predict):       [7, 8, 9, 2]

src    = torch.tensor([src_ids])          # [B, S]  (extra [] creates the batch dim)
tgt    = torch.tensor([tgt_input_ids])    # [B, T]
labels = torch.tensor([tgt_label_ids])    # [B, T]

print("vocab_size:", vocab_size)
print("src    :", src.tolist(),    [id_to_token[i] for i in src_ids])
print("tgt in :", tgt.tolist(),    [id_to_token[i] for i in tgt_input_ids])
print("labels :", labels.tolist(), [id_to_token[i] for i in tgt_label_ids])
print("shapes :", tuple(src.shape), tuple(tgt.shape), tuple(labels.shape))

vocab_size: 10
src    : [[4, 5, 6, 2]] ['i', 'love', 'cats', '<eos>']
tgt in : [[1, 7, 8, 9]] ['<bos>', "j'aime", 'les', 'chats']
labels : [[7, 8, 9, 2]] ["j'aime", 'les', 'chats', '<eos>']
shapes : (1, 4) (1, 4) (1, 4)


## 3. Embeddings and positional information
`nn.Embedding(vocab_size, D)` is a learnable lookup table `[vocab_size, D]`: `[B, L] → [B, L, D]`.
Attention alone ignores order, so we add sinusoidal position vectors:

$PE_{(pos,2i)}=\sin(pos/10000^{2i/D}),\quad PE_{(pos,2i+1)}=\cos(pos/10000^{2i/D})$

In [3]:
def sinusoidal_positions(max_len, d_model):
    position = torch.arange(max_len).float().unsqueeze(1)           # [L, 1]
    two_i = torch.arange(0, d_model, 2).float()                     # [D/2] = 0,2,4,6
    div_term = torch.exp(two_i * (-math.log(10000.0) / d_model))    # [D/2] = 10000^(-2i/D)
    pe = torch.zeros(max_len, d_model)                              # [L, D]
    pe[:, 0::2] = torch.sin(position * div_term)                    # even dims
    pe[:, 1::2] = torch.cos(position * div_term)                    # odd dims
    return pe

pe = sinusoidal_positions(50, D)
print("pe shape:", tuple(pe.shape))
print("pe[0] (sin(0)=0, cos(0)=1 pattern):", pe[0].tolist())

embedding = nn.Embedding(vocab_size, D)          # weight [10, 8]
src_emb = embedding(src)                         # [B,S] -> [B,S,D]
src_x = src_emb * math.sqrt(D) + pe[:S]          # [B,S,D] + [S,D] broadcasts over batch
tgt_x = embedding(tgt) * math.sqrt(D) + pe[:T]   # [B,T,D]

print("src_emb:", tuple(src_emb.shape), "| src_x:", tuple(src_x.shape), "| tgt_x:", tuple(tgt_x.shape))

pe shape: (50, 8)
pe[0] (sin(0)=0, cos(0)=1 pattern): [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
src_emb: (1, 4, 8) | src_x: (1, 4, 8) | tgt_x: (1, 4, 8)


## 4. Scaled dot-product attention

$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$

Mask convention in this notebook: **`True` = allowed**, `False` = blocked.

In [4]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)                                  # last dim = per-head size
    scores = Q @ K.transpose(-2, -1)                  # [..., Lq, d_k] @ [..., d_k, Lk] -> [..., Lq, Lk]
    scores = scores / math.sqrt(d_k)                  # keep scores at unit scale
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))   # blocked -> -inf -> weight 0
    weights = torch.softmax(scores, dim=-1)           # softmax over KEYS: each row sums to 1
    output = weights @ V                              # [..., Lq, Lk] @ [..., Lk, d_v] -> [..., Lq, d_v]
    return output, weights

# Hand-checkable toy example (d_k = 2), same numbers as the lesson.
Qt = torch.tensor([[[1., 0.], [0., 1.], [1., 1.], [-1., 1.]]])
Kt = torch.tensor([[[1., 0.], [0., 1.], [1., 1.], [1., -1.]]])
Vt = torch.tensor([[[1., 0.], [0., 1.], [2., 2.], [0., 0.]]])

print("raw scores Q K^T:\n", (Qt @ Kt.transpose(-2, -1))[0])
out_t, w_t = scaled_dot_product_attention(Qt, Kt, Vt)
print("weights (rows sum to 1):\n", w_t[0].round(decimals=3))
print("output:\n", out_t[0].round(decimals=3))

expected_w = torch.tensor([[.286,.141,.286,.286],[.180,.365,.365,.089],
                           [.221,.221,.449,.109],[.131,.539,.266,.065]])
expected_o = torch.tensor([[.859,.714],[.911,1.096],[1.118,1.118],[.662,1.070]])
print("matches hand computation:", torch.allclose(w_t[0], expected_w, atol=2e-3),
      torch.allclose(out_t[0], expected_o, atol=3e-3))

# With a causal mask the upper triangle must be exactly 0
_, w_masked = scaled_dot_product_attention(Qt, Kt, Vt, mask=torch.tril(torch.ones(4, 4)).bool())
print("causal-masked weights:\n", w_masked[0].round(decimals=3))

raw scores Q K^T:
 tensor([[ 1.,  0.,  1.,  1.],
        [ 0.,  1.,  1., -1.],
        [ 1.,  1.,  2.,  0.],
        [-1.,  1.,  0., -2.]])
weights (rows sum to 1):
 tensor([[0.2860, 0.1410, 0.2860, 0.2860],
        [0.1800, 0.3650, 0.3650, 0.0890],
        [0.2210, 0.2210, 0.4490, 0.1090],
        [0.1310, 0.5390, 0.2660, 0.0650]])
output:
 tensor([[0.8590, 0.7140],
        [0.9110, 1.0960],
        [1.1180, 1.1180],
        [0.6620, 1.0700]])
matches hand computation: True True
causal-masked weights:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.3300, 0.6700, 0.0000, 0.0000],
        [0.2480, 0.2480, 0.5030, 0.0000],
        [0.1310, 0.5390, 0.2660, 0.0650]])


## 5. Masks
* **Causal mask** (`torch.tril`): position *i* may see keys *j ≤ i*. Hides the future.
* **Padding mask**: hides `<pad>` keys. Shape `[B,1,1,L]` so it broadcasts over heads and queries.

In [5]:
def make_causal_mask(seq_len, device=None):
    ones = torch.ones(seq_len, seq_len, device=device)
    return torch.tril(ones).bool()                    # True on/below the diagonal

print("causal mask (T=4):\n", make_causal_mask(4))

ids = torch.tensor([[7, 8, 0, 0]])                    # two real tokens, two pads
pad_mask = (ids != PAD_ID).unsqueeze(1).unsqueeze(2)  # [B,L] -> [B,1,1,L]
print("padding mask shape:", tuple(pad_mask.shape), "values:", pad_mask.flatten().tolist())

combined = make_causal_mask(4) & pad_mask             # [4,4] & [1,1,1,4] -> [1,1,4,4]
print("causal & padding shape:", tuple(combined.shape))
print(combined[0, 0].int())

causal mask (T=4):
 tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])
padding mask shape: (1, 1, 1, 4) values: [True, True, False, False]
causal & padding shape: (1, 1, 4, 4)
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 0, 0]], dtype=torch.int32)


## 6. Multi-head attention
`[B,L,D] → reshape [B,L,H,D_HEAD] → permute [B,H,L,D_HEAD]`, attention per head, then merge back and apply `W_O`.
`forward(query_input, key_value_input)` lets one class serve self-attention *and* cross-attention.

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.h = num_heads
        self.d_head = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):                         # [B, L, D] -> [B, H, L, D_head]
        Bsz, L, _ = x.shape
        x = x.reshape(Bsz, L, self.h, self.d_head)
        return x.permute(0, 2, 1, 3)

    def merge_heads(self, x):                         # [B, H, L, D_head] -> [B, L, D]
        Bsz, Hh, L, Dh = x.shape
        x = x.permute(0, 2, 1, 3)                     # [B, L, H, D_head]
        return x.reshape(Bsz, L, Hh * Dh)             # concatenate heads

    def forward(self, query_input, key_value_input, mask=None):
        Q = self.split_heads(self.w_q(query_input))       # [B, H, Lq, D_head]
        K = self.split_heads(self.w_k(key_value_input))   # [B, H, Lk, D_head]
        V = self.split_heads(self.w_v(key_value_input))   # [B, H, Lk, D_head]
        out, weights = scaled_dot_product_attention(Q, K, V, mask)  # [B,H,Lq,D_head]
        out = self.merge_heads(out)                       # [B, Lq, D]
        return self.w_o(out), weights                     # [B, Lq, D], [B,H,Lq,Lk]

# reshape/permute demo: values 0..7 -> head 0 gets 0..3, head 1 gets 4..7
t = torch.arange(8).reshape(1, 1, 8)
print("head split demo:", t.reshape(1, 1, 2, 4).permute(0, 2, 1, 3).tolist())

mha = MultiHeadAttention(D, H)
mha_out, mha_w = mha(src_x, src_x)
print("MHA out:", tuple(mha_out.shape), "| weights:", tuple(mha_w.shape))

# cross-attention shapes: target length (3) != source length (4)
c_out, c_w = MultiHeadAttention(D, H)(torch.randn(1, 3, D), torch.randn(1, 4, D))
print("cross-attn out:", tuple(c_out.shape), "| weights [B,H,T,S]:", tuple(c_w.shape))

head split demo: [[[[0, 1, 2, 3]], [[4, 5, 6, 7]]]]
MHA out: (1, 4, 8) | weights: (1, 2, 4, 4)
cross-attn out: (1, 3, 8) | weights [B,H,T,S]: (1, 2, 3, 4)


## 7. Feed-forward network and the Encoder
Post-LN sublayer pattern: `x = LayerNorm(x + Sublayer(x))`. Residual needs identical shapes `[B,L,D]`.

In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)      # expand  D -> D_FF
        self.activation = nn.ReLU()                  # non-linearity
        self.linear2 = nn.Linear(d_ff, d_model)      # project D_FF -> D

    def forward(self, x):                            # applied independently at each position
        return self.linear2(self.activation(self.linear1(x)))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, src_mask=None):                     # x: [B, S, D]
        attn_out, attn_w = self.self_attn(x, x, mask=src_mask)   # Q,K,V all from x
        x = self.norm1(x + attn_out)                         # residual + LayerNorm
        x = self.norm2(x + self.ffn(x))                      # residual + LayerNorm
        return x, attn_w


class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])

    def forward(self, x, src_mask=None):
        for layer in self.layers:
            x, _ = layer(x, src_mask)
        return x                                             # [B, S, D]

enc_test = Encoder(D, H, D_FF, num_layers=2)
print("Encoder output shape:", tuple(enc_test(src_x).shape), "(same as input)")

Encoder output shape: (1, 4, 8) (same as input)


## 8. The Decoder
Per layer: masked self-attention → cross-attention (Q from decoder, K,V from encoder output) → FFN, each with residual + LayerNorm.

In [8]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)    # masked
        self.cross_attn = MultiHeadAttention(d_model, num_heads)   # separate weights
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, enc_out, tgt_mask=None, src_mask=None):
        sa_out, sa_w = self.self_attn(x, x, mask=tgt_mask)           # [B,T,D]
        x = self.norm1(x + sa_out)
        ca_out, ca_w = self.cross_attn(x, enc_out, mask=src_mask)    # Q: x [B,T,D]; K,V: enc_out [B,S,D]
        x = self.norm2(x + ca_out)                                   # output length = T (query length)
        x = self.norm3(x + self.ffn(x))
        return x, sa_w, ca_w


class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)])

    def forward(self, x, enc_out, tgt_mask=None, src_mask=None):
        for layer in self.layers:              # every layer reads the SAME encoder output
            x, _, _ = layer(x, enc_out, tgt_mask, src_mask)
        return x                               # [B, T, D]

## 9. The complete Transformer

In [9]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=8, num_heads=2, d_ff=16,
                 num_layers=2, max_len=50, pad_id=0):
        super().__init__()
        self.d_model = d_model
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, d_model)            # shared src/tgt
        self.register_buffer("pe", sinusoidal_positions(max_len, d_model))  # not trained
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)             # D -> vocab

    def embed(self, ids):                                  # [B, L] -> [B, L, D]
        x = self.embedding(ids) * math.sqrt(self.d_model)
        return x + self.pe[: ids.size(1)]

    def encode(self, src):                                 # src: [B, S]
        src_mask = (src != self.pad_id).unsqueeze(1).unsqueeze(2)     # [B,1,1,S]
        enc_out = self.encoder(self.embed(src), src_mask)             # [B,S,D]
        return enc_out, src_mask

    def make_tgt_mask(self, tgt):                          # causal AND not-padding
        causal = make_causal_mask(tgt.size(1), device=tgt.device)     # [T,T]
        pad = (tgt != self.pad_id).unsqueeze(1).unsqueeze(2)          # [B,1,1,T]
        return causal & pad                                           # [B,1,T,T]

    def decode(self, tgt, enc_out, src_mask):              # tgt: [B, T]
        tgt_mask = self.make_tgt_mask(tgt)
        dec_out = self.decoder(self.embed(tgt), enc_out, tgt_mask, src_mask)  # [B,T,D]
        return self.output_proj(dec_out)                              # logits [B,T,vocab]

    def forward(self, src, tgt):
        enc_out, src_mask = self.encode(src)
        return self.decode(tgt, enc_out, src_mask)

torch.manual_seed(0)
model = Transformer(vocab_size, D, H, D_FF, num_layers=2)
n_params = sum(p.numel() for p in model.parameters())
print("trainable parameters:", n_params)

trainable parameters: 2986


## 10. First forward pass
Weights are random, so predictions are meaningless. That is expected. We only check shapes and probability properties.

In [10]:
logits = model(src, tgt)                     # [B, T, vocab_size]
probs = torch.softmax(logits, dim=-1)        # softmax over the vocabulary axis

print("logits:", tuple(logits.shape), "| probs:", tuple(probs.shape))
print("each position sums to 1:", probs.sum(dim=-1))
pred_ids = probs.argmax(dim=-1)              # token SELECTION is a separate decision from softmax
print("argmax IDs (random model):", pred_ids.tolist(),
      [id_to_token[i] for i in pred_ids[0].tolist()])

logits: (1, 4, 10) | probs: (1, 4, 10)
each position sums to 1: tensor([[1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SumBackward1>)
argmax IDs (random model): [[7, 0, 5, 7]] ["j'aime", '<pad>', 'love', "j'aime"]


## 11. Manual trace of the first Encoder layer (shapes at every step)

In [11]:
def show(name, t):
    print(f"{name:<24}{tuple(t.shape)}")

model.eval()
with torch.no_grad():
    x = model.embed(src);                                   show("encoder input", x)
    layer = model.encoder.layers[0]
    attn = layer.self_attn
    Q = attn.split_heads(attn.w_q(x));                      show("Q (split heads)", Q)
    K = attn.split_heads(attn.w_k(x));                      show("K", K)
    V = attn.split_heads(attn.w_v(x));                      show("V", V)
    scores = Q @ K.transpose(-2, -1);                       show("scores Q K^T", scores)
    weights = torch.softmax(scores / math.sqrt(attn.d_head), dim=-1); show("weights", weights)
    heads = weights @ V;                                    show("per-head output", heads)
    merged = attn.merge_heads(heads);                       show("merged heads", merged)
    x1 = layer.norm1(x + attn.w_o(merged));                 show("after Add & LN 1", x1)
    hidden = layer.ffn.activation(layer.ffn.linear1(x1));   show("FFN hidden", hidden)
    x2 = layer.norm2(x1 + layer.ffn.linear2(hidden));       show("layer output", x2)

    # sanity: manual trace equals the real layer
    real, _ = layer(x, model.encode(src)[1])
    print("manual == layer output:", torch.allclose(x2, real, atol=1e-6))

encoder input           (1, 4, 8)
Q (split heads)         (1, 2, 4, 4)
K                       (1, 2, 4, 4)
V                       (1, 2, 4, 4)
scores Q K^T            (1, 2, 4, 4)
weights                 (1, 2, 4, 4)
per-head output         (1, 2, 4, 4)
merged heads            (1, 4, 8)
after Add & LN 1        (1, 4, 8)
FFN hidden              (1, 4, 16)
layer output            (1, 4, 8)
manual == layer output: True


## 12. Inspect the three attention patterns

In [12]:
with torch.no_grad():
    enc_out, src_mask = model.encode(src)
    tgt_mask = model.make_tgt_mask(tgt)
    dec_layer = model.decoder.layers[0]
    x_t = model.embed(tgt)
    _, sa_w, ca_w = dec_layer(x_t, enc_out, tgt_mask, src_mask)
    _, enc_w = model.encoder.layers[0](model.embed(src), src_mask)

print("Encoder self-attn weights  [B,H,S,S]:", tuple(enc_w.shape))
print("Decoder masked self-attn   [B,H,T,T]:", tuple(sa_w.shape))
print("Decoder cross-attn         [B,H,T,S]:", tuple(ca_w.shape))
print("\nmasked self-attn, head 0 (upper triangle must be exactly 0):\n", sa_w[0, 0].round(decimals=3))
print("\ncross-attn, head 0 (rows = target positions, cols = source positions):\n", ca_w[0, 0].round(decimals=3))
print("\nrows sum to 1:", torch.allclose(ca_w.sum(-1), torch.ones(1, H, T)))

# S != T check: source length 5, target length 4 -> cross weights [B,H,4,5]
src5 = torch.tensor([[4, 5, 6, 5, 2]])
with torch.no_grad():
    e5, m5 = model.encode(src5)
    _, _, ca5 = dec_layer(x_t, e5, tgt_mask, m5)
print("\nS=5, T=4 cross-attn weights:", tuple(ca5.shape))

Encoder self-attn weights  [B,H,S,S]: (1, 2, 4, 4)
Decoder masked self-attn   [B,H,T,T]: (1, 2, 4, 4)
Decoder cross-attn         [B,H,T,S]: (1, 2, 4, 4)

masked self-attn, head 0 (upper triangle must be exactly 0):
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5100, 0.4900, 0.0000, 0.0000],
        [0.5970, 0.0010, 0.4020, 0.0000],
        [0.9340, 0.0150, 0.0470, 0.0030]])

cross-attn, head 0 (rows = target positions, cols = source positions):
 tensor([[0.4260, 0.4070, 0.1030, 0.0640],
        [0.2040, 0.3370, 0.2090, 0.2510],
        [0.2870, 0.2570, 0.2040, 0.2510],
        [0.3260, 0.2040, 0.2220, 0.2470]])

rows sum to 1: True

S=5, T=4 cross-attn weights: (1, 2, 4, 5)


## 13. Training (minimal)
Forward → loss → `backward()` (gradients) → `optimizer.step()` (update).
This **overfits one sentence pair** to show the machinery works. It does not make a translator.
`CrossEntropyLoss` expects raw logits (it applies log-softmax internally); do not softmax first.

In [13]:
torch.manual_seed(0)
model = Transformer(vocab_size, D, H, D_FF, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

print("uniform-guess loss ln(10) =", round(math.log(vocab_size), 4))
model.train()
for step in range(1, 301):
    logits = model(src, tgt)                                              # forward
    loss = criterion(logits.reshape(-1, vocab_size), labels.reshape(-1))  # [B*T,vocab] vs [B*T]
    optimizer.zero_grad()                                                 # clear old gradients
    loss.backward()                                                       # backpropagation
    optimizer.step()                                                      # parameter update
    if step == 1 or step % 50 == 0:
        print(f"step {step:>3} | loss {loss.item():.4f}")

uniform-guess loss ln(10) = 2.3026
step   1 | loss 1.8868


step  50 | loss 0.0268


step 100 | loss 0.0072


step 150 | loss 0.0038


step 200 | loss 0.0024


step 250 | loss 0.0017


step 300 | loss 0.0012


## 14. Autoregressive generation (greedy)
Run the Encoder **once**, then feed the Decoder its own predictions, one token at a time.

In [14]:
@torch.no_grad()
def greedy_generate(model, src, max_len=6):
    model.eval()
    enc_out, src_mask = model.encode(src)                 # Encoder runs ONCE
    ys = torch.tensor([[BOS_ID]])                         # decoder input so far: [1, 1]
    for _ in range(max_len):
        logits = model.decode(ys, enc_out, src_mask)      # [1, len(ys), vocab]
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)   # last position -> [1, 1]
        ys = torch.cat([ys, next_id], dim=1)              # append, shape [1, len+1]
        print("  step -> predicted:", id_to_token[next_id.item()])
        if next_id.item() == EOS_ID:
            break
    return ys

print("Generating from:", [id_to_token[i] for i in src_ids])
out_ids = greedy_generate(model, src)[0].tolist()
print("Result:", [id_to_token[i] for i in out_ids])

Generating from: ['i', 'love', 'cats', '<eos>']
  step -> predicted: j'aime
  step -> predicted: les
  step -> predicted: chats
  step -> predicted: <eos>
Result: ['<bos>', "j'aime", 'les', 'chats', '<eos>']


## Simplifications (vs. production models)
No dropout; no biases in attention projections; Post-LN (modern LLMs usually use Pre-LN); sinusoidal positions and ReLU;
shared embedding, untied output layer; whitespace tokenizer; greedy decoding without a KV cache; tiny dimensions.

**Note:** in PyTorch's built-in `nn.MultiheadAttention`, boolean masks use `True = blocked`, the opposite of this notebook.